# LSTM Model Evaluation Notebook
Load a trained model checkpoint and visualize predictions on validation data (Site B)

## 1. Configuration
Update these settings to match your model and data paths

In [ ]:
model_number = 3  # Change this to switch between different model configurations

if model_number == 1:
    # Best model from basic regression notebook run
    MODEL_CHECKPOINT = "C:/Github_FabianDubach/aicomp-flextrack/_test-folder/best_model.pt"
    MODEL_CONFIG = {
        'input_size': 37,      # Number of features
        'hidden_size': 64,     # Hidden layer size from best run
        'num_layers': 2,       # Number of LSTM layers from best run
        'output_size': 1,      # Output size
        'dropout': 0.3,        # Dropout from best run
        'sequence_length': 12  # Sequence length
    }
elif model_number == 2:
    # Model run try with penalized loss for predictions on 0
    MODEL_CHECKPOINT = "C:/Github_FabianDubach/aicomp-flextrack/wandb/run-20251120_174141-10m6a29w/files/best_model.pt"
    MODEL_CONFIG = {
        'input_size': 37,      # Number of features
        'hidden_size': 64,     # Hidden layer size from best run
        'num_layers': 2,       # Number of LSTM layers from best run
        'output_size': 1,      # Output size
        'dropout': 0.3,        # Dropout from best run
        'sequence_length': 12  # Sequence length
    }
elif model_number == 3:
    # Model run from training only on events
    MODEL_CHECKPOINT = "C:/Github_FabianDubach/aicomp-flextrack/wandb/run-20251127_204031-x49xw25h/files/best_model.pt"
    MODEL_CONFIG = {
        'input_size': 37,      # Number of features
        'hidden_size': 64,     # Hidden layer size from best run
        'num_layers': 2,       # Number of LSTM layers from best run
        'output_size': 1,      # Output size
        'dropout': 0.4,        # Dropout from best run
        'sequence_length': 48  # Sequence length
    }

# Data paths
DATA_PATH = "C:/Github_FabianDubach/aicomp-flextrack/data/regression/"

# Constants
ENTRIES_PER_DAY = 53  # 15 min intervals from 6:00 to 19:00

## 2. Imports and Setup

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
import holidays
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('default')
sns.set_palette('husl')

# Add path for comp_metrics
sys.path.append('C:/Github_FabianDubach/aicomp-flextrack/src/evaluation')
from comp_metrics import evaluate_all_metrics

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 3. Model Definition
Must match your training script exactly

In [ ]:
class SimpleRNN(nn.Module):
    """LSTM model for regression"""
    def __init__(self, input_size, hidden_size, num_layers, output_size, dropout=0.3):
        super(SimpleRNN, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, 
                          batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        
        out, _ = self.lstm(x, (h0, c0))
        out = out[:, -1, :]
        out = self.dropout(out)
        out = self.fc(out)
        return out

print("✅ Model class defined")

## 4. Data Processing Functions

In [ ]:
def add_time_features(df):
    """Add temporal features to the dataframe"""
    australian_holidays = holidays.AU()
    
    df['Timestamp_Local'] = pd.to_datetime(df['Timestamp_Local'])
    df['hour'] = df['Timestamp_Local'].dt.hour
    df['minute'] = df['Timestamp_Local'].dt.minute
    df['month'] = df['Timestamp_Local'].dt.month
    df['day_of_week'] = df['Timestamp_Local'].dt.dayofweek
    df['is_weekend'] = df['Timestamp_Local'].dt.dayofweek >= 5
    df['is_holiday'] = df['Timestamp_Local'].dt.date.apply(lambda x: x in australian_holidays)
    return df


def engineer_features(df):
    """Complete feature engineering pipeline"""
    df = add_time_features(df)
    
    # Remove night entries (6 AM to 7 PM only)
    df = df[
        ((df['hour'] > 6) & (df['hour'] < 19)) |
        ((df['hour'] == 6) & (df['minute'] >= 0)) |
        ((df['hour'] == 19) & (df['minute'] == 0))
    ].copy()
    
    # Cyclic encoding for month
    df['month_sin'] = np.sin(2 * np.pi * (df['month'] / 12))
    df['month_cos'] = np.cos(2 * np.pi * (df['month'] / 12))
    df.drop(columns=['month'], inplace=True)
    
    # Difference features
    df['Building_Power_kW_diff_15min'] = df.groupby('Site')['Building_Power_kW'].diff(1)
    df['Building_Power_kW_diff_1h'] = df.groupby('Site')['Building_Power_kW'].diff(4)
    df['Building_Power_kW_diff_1d'] = df.groupby('Site')['Building_Power_kW'].diff(ENTRIES_PER_DAY)
    df['Dry_Bulb_Temperature_C_diff_15min'] = df.groupby('Site')['Dry_Bulb_Temperature_C'].diff(1)
    df['Global_Horizontal_Radiation_W/m2_diff_15min'] = df.groupby('Site')['Global_Horizontal_Radiation_W/m2'].diff(1)
    
    # Rolling statistics
    group = df.groupby('Site')['Building_Power_kW']
    df['Building_Power_kW_rolling_mean_1h'] = group.rolling(4).mean().reset_index(level=0, drop=True)
    df['Building_Power_kW_rolling_mean_2h'] = group.rolling(8).mean().reset_index(level=0, drop=True)
    df['Building_Power_kW_rolling_mean_1d'] = group.rolling(ENTRIES_PER_DAY).mean().reset_index(level=0, drop=True)
    df['Building_Power_kW_rolling_std_1h'] = group.rolling(4).std().reset_index(level=0, drop=True)
    df['Building_Power_kW_rolling_std_2h'] = group.rolling(8).std().reset_index(level=0, drop=True)
    df['Building_Power_kW_rolling_std_1d'] = group.rolling(ENTRIES_PER_DAY).std().reset_index(level=0, drop=True)
    df['Building_Power_kW_rolling_min_1h'] = group.rolling(4).min().reset_index(level=0, drop=True)
    df['Building_Power_kW_rolling_min_2h'] = group.rolling(8).min().reset_index(level=0, drop=True)
    df['Building_Power_kW_rolling_max_1h'] = group.rolling(4).max().reset_index(level=0, drop=True)
    df['Building_Power_kW_rolling_max_2h'] = group.rolling(8).max().reset_index(level=0, drop=True)
    
    # One-hot encodings
    df = pd.get_dummies(df, columns=['minute', 'day_of_week'])
    df = pd.get_dummies(df, columns=['Demand_Response_Flag'])
    df['is_holiday'] = df['is_holiday'].astype(int)
    df['is_weekend'] = df['is_weekend'].astype(int)
    
    return df

print("✅ Data processing functions defined")

## 5. Load and Process Data
### This section now matches the test_lstm_regression.ipynb approach:
- Split data by site FIRST (Site A, B, C)
- Scale each site SEPARATELY
- Train on Sites A + C, Validate on Site B

In [ ]:
print("Loading data...")
df_train = pd.read_csv(f"{DATA_PATH}regression-train.csv")

print("Engineering features...")
df_train = engineer_features(df_train)

print(f"Total samples after feature engineering: {len(df_train)}")

In [ ]:
# Define feature columns
continuous_feature_columns = [
    'Dry_Bulb_Temperature_C', 'Global_Horizontal_Radiation_W/m2', 'Building_Power_kW',
    'Building_Power_kW_diff_15min', 'Building_Power_kW_diff_1h', 'Building_Power_kW_diff_1d',
    'Dry_Bulb_Temperature_C_diff_15min', 'Global_Horizontal_Radiation_W/m2_diff_15min',
    'Building_Power_kW_rolling_mean_1h', 'Building_Power_kW_rolling_mean_2h', 'Building_Power_kW_rolling_mean_1d',
    'Building_Power_kW_rolling_std_1h', 'Building_Power_kW_rolling_std_2h', 'Building_Power_kW_rolling_std_1d',
    'Building_Power_kW_rolling_min_1h', 'Building_Power_kW_rolling_min_2h', 
    'Building_Power_kW_rolling_max_1h', 'Building_Power_kW_rolling_max_2h', 'hour'
]

categorical_feature_columns = [
    'minute_0', 'minute_15', 'minute_30', 'minute_45',
    'day_of_week_0', 'day_of_week_1', 'day_of_week_2', 'day_of_week_3', 
    'day_of_week_4', 'day_of_week_5', 'day_of_week_6',
    'Demand_Response_Flag_-1', 'Demand_Response_Flag_0', 'Demand_Response_Flag_1',
    'is_weekend', 'is_holiday'
]

cyclic_feature_columns = ['month_sin', 'month_cos']
target_column = 'Demand_Response_Capacity_kW'

print(f"Feature dimensions:")
print(f"  Continuous: {len(continuous_feature_columns)}")
print(f"  Categorical: {len(categorical_feature_columns)}")
print(f"  Cyclic: {len(cyclic_feature_columns)}")
print(f"  Total: {len(continuous_feature_columns) + len(categorical_feature_columns) + len(cyclic_feature_columns)}")

### Split data by site (matching test_lstm_regression)

In [ ]:
# Split dataframe into sites
df_train_site_a = df_train[0:19345]
df_train_site_b = df_train[19345:38690]
df_train_site_c = df_train[38690:58035]

print(f"Site A samples: {len(df_train_site_a)}")
print(f"Site B samples: {len(df_train_site_b)}")
print(f"Site C samples: {len(df_train_site_c)}")

In [ ]:
# Extract features for each site
X_continuous_site_a = df_train_site_a[continuous_feature_columns].values
X_continuous_site_b = df_train_site_b[continuous_feature_columns].values
X_continuous_site_c = df_train_site_c[continuous_feature_columns].values

X_categorical_site_a = df_train_site_a[categorical_feature_columns].values
X_categorical_site_b = df_train_site_b[categorical_feature_columns].values
X_categorical_site_c = df_train_site_c[categorical_feature_columns].values

X_cyclic_site_a = df_train_site_a[cyclic_feature_columns].values
X_cyclic_site_b = df_train_site_b[cyclic_feature_columns].values
X_cyclic_site_c = df_train_site_c[cyclic_feature_columns].values

y_site_a = df_train_site_a[target_column].values.reshape(-1, 1)
y_site_b = df_train_site_b[target_column].values.reshape(-1, 1)
y_site_c = df_train_site_c[target_column].values.reshape(-1, 1)

print("Feature extraction complete!")

### Scale each site separately

In [ ]:
# Create separate scalers for each site
scaler_X_site_a = StandardScaler()
scaler_X_site_b = StandardScaler()
scaler_X_site_c = StandardScaler()

scaler_y_site_a = StandardScaler()
scaler_y_site_b = StandardScaler()
scaler_y_site_c = StandardScaler()

# Fit and transform each site separately
X_scaled_site_a = scaler_X_site_a.fit_transform(X_continuous_site_a)
X_scaled_site_b = scaler_X_site_b.fit_transform(X_continuous_site_b)
X_scaled_site_c = scaler_X_site_c.fit_transform(X_continuous_site_c)

y_site_a = scaler_y_site_a.fit_transform(y_site_a)
y_site_b = scaler_y_site_b.fit_transform(y_site_b)
y_site_c = scaler_y_site_c.fit_transform(y_site_c)

print("Separate scaling complete for each site!")

In [ ]:
# Concatenate scaled continuous with unscaled categorical and cyclic features
X_site_a = np.concatenate([X_scaled_site_a, X_categorical_site_a, X_cyclic_site_a], axis=1)
X_site_b = np.concatenate([X_scaled_site_b, X_categorical_site_b, X_cyclic_site_b], axis=1)
X_site_c = np.concatenate([X_scaled_site_c, X_categorical_site_c, X_cyclic_site_c], axis=1)

# Convert to float32
X_site_a = X_site_a.astype(np.float32)
X_site_b = X_site_b.astype(np.float32)
X_site_c = X_site_c.astype(np.float32)

y_site_a = y_site_a.astype(np.float32)
y_site_b = y_site_b.astype(np.float32)
y_site_c = y_site_c.astype(np.float32)

print(f"Site A feature shape: {X_site_a.shape}")
print(f"Site B feature shape: {X_site_b.shape}")
print(f"Site C feature shape: {X_site_c.shape}")

### Remove incomplete entries (first day of each site)

In [ ]:
# Create masks to exclude first ENTRIES_PER_DAY of each site
mask_site_a = np.ones(len(X_site_a), dtype=bool)
mask_site_b = np.ones(len(X_site_b), dtype=bool)
mask_site_c = np.ones(len(X_site_c), dtype=bool)

mask_site_a[0:ENTRIES_PER_DAY] = False
mask_site_b[0:ENTRIES_PER_DAY] = False
mask_site_c[0:ENTRIES_PER_DAY] = False

# Apply masks
X_site_a = X_site_a[mask_site_a]
X_site_b = X_site_b[mask_site_b]
X_site_c = X_site_c[mask_site_c]

y_site_a = y_site_a[mask_site_a]
y_site_b = y_site_b[mask_site_b]
y_site_c = y_site_c[mask_site_c]

print(f"After removing incomplete entries:")
print(f"  Site A: {len(X_site_a)} samples")
print(f"  Site B: {len(X_site_b)} samples")
print(f"  Site C: {len(X_site_c)} samples")

### Split: Train on Sites A + C, Validate on Site B

In [ ]:
# Train on Site A and Site C, validate on Site B
X_train = np.vstack((X_site_a, X_site_c))
X_val = X_site_b
y_train = np.vstack((y_site_a, y_site_c))
y_val = y_site_b

print(f"Training samples: {len(X_train)}")
print(f"Validation samples (Site B): {len(X_val)}")

### Create sequences

In [ ]:
def create_sequences(X, y, seq_length):
    sequences_X = []
    sequences_y = []
    
    for i in range(len(X) - seq_length):
        sequences_X.append(X[i:i+seq_length])
        sequences_y.append(y[i+seq_length - 1])
    
    return np.array(sequences_X), np.array(sequences_y)

sequence_length = MODEL_CONFIG['sequence_length']
X_train_seq, y_train_seq = create_sequences(X_train, y_train, sequence_length)
X_val_seq, y_val_seq = create_sequences(X_val, y_val, sequence_length)

print(f"\n✅ Data loaded and processed:")
print(f"   Training sequences: {len(X_train_seq)}")
print(f"   Validation sequences (Site B): {len(X_val_seq)}")
print(f"   Input features: {X_val_seq.shape[2]}")
print(f"   Sequence length: {X_val_seq.shape[1]}")

### Extract metadata for evaluation

In [ ]:
# Extract building power (unscale using Site B's scaler)
val_building_power_scaled = X_val_seq[:, -1, 2]
temp = np.zeros((len(val_building_power_scaled), 19))
temp[:, 2] = val_building_power_scaled
val_building_power = scaler_X_site_b.inverse_transform(temp)[:, 2]

# Extract demand response flags
val_demand_flags = np.argmax(X_val_seq[:, -1, 30:33], axis=1) - 1

# All validation samples are from Site B
val_sites = np.array(['siteB'] * len(X_val_seq))

print("✅ Metadata extracted:")
print(f"   Demand flags unique: {np.unique(val_demand_flags)} (expected: [-1, 0, 1])")
print(f"   Building power range: [{val_building_power.min():.2f}, {val_building_power.max():.2f}] kW")
print(f"   Site: Site B (Validation set)")

## 6. Load Model and Make Predictions

In [ ]:
print("Loading model...")
model = SimpleRNN(
    input_size=MODEL_CONFIG['input_size'],
    hidden_size=MODEL_CONFIG['hidden_size'],
    num_layers=MODEL_CONFIG['num_layers'],
    output_size=MODEL_CONFIG['output_size'],
    dropout=MODEL_CONFIG['dropout']
).to(device)

# Load checkpoint
model.load_state_dict(torch.load(MODEL_CHECKPOINT, map_location=device))
model.eval()
print("✅ Model loaded successfully")

In [ ]:
print("Making predictions on Site B validation set...")
with torch.no_grad():
    X_val_tensor = torch.FloatTensor(X_val_seq).to(device)
    predictions_scaled = model(X_val_tensor).cpu().numpy()

# Inverse transform using Site B's scaler
predictions = scaler_y_site_b.inverse_transform(predictions_scaled).flatten()
targets = scaler_y_site_b.inverse_transform(y_val_seq).flatten()

targets[np.abs(targets) <= 0.00001] = 0


print(f"✅ Predictions complete:")
print(f"   Samples: {len(predictions)}")
print(f"   Prediction range: [{predictions.min():.2f}, {predictions.max():.2f}] kW")
print(f"   Target range: [{targets.min():.2f}, {targets.max():.2f}] kW")

In [ ]:
# ------------------------------
# 1️⃣ Align masked validation dataframe with predictions
# ------------------------------
df_val_masked = df_train_site_b[mask_site_b].reset_index(drop=True)

# Now each prediction corresponds to row i + sequence_length - 1
df_val_aligned = df_val_masked.iloc[sequence_length - 1 : sequence_length - 1 + len(predictions)].reset_index(drop=True)

# ------------------------------
# 2️⃣ Extract DR flags from aligned dataframe
# ------------------------------
dr_flags = df_val_aligned[['Demand_Response_Flag_-1', 
                           'Demand_Response_Flag_0', 
                           'Demand_Response_Flag_1']]

# Boolean mask: True where there is NO DR event
no_dr_mask = dr_flags['Demand_Response_Flag_0'] == 1

# ------------------------------
# 3️⃣ Zero out predictions where no DR event
# ------------------------------
predictions_corrected = predictions.copy()
predictions_corrected[no_dr_mask.values] = 0

In [ ]:
print(targets[525:550])
print(predictions_corrected[525:550])

## 7. Evaluate Metrics

In [ ]:
# Calculate comprehensive metrics using comp_metrics
metrics = evaluate_all_metrics(
    y_true=targets,                     # unchanged targets
    y_pred=predictions_corrected,       # zeroed predictions
    site_labels=val_sites,
    building_power=val_building_power,
    demand_flags=val_demand_flags
)

# Calculate basic metrics
mae = np.abs(targets - predictions).mean()
rmse = np.sqrt(np.mean((targets - predictions) ** 2))
r2 = 1 - (np.sum((targets - predictions) ** 2) / np.sum((targets - targets.mean()) ** 2))

print("\n" + "=" * 80)
print("EVALUATION METRICS - SITE B")
print("=" * 80)
print(f"\nCompetition Metrics:")
print(f"  NMAE (range): {metrics['nmae_range']:.2f}%")
print(f"  NMAE (mean):  {metrics['nmae_mean']:.2f}%")
print(f"  NRMSE (range): {metrics['nrmse_range']:.2f}%")
print(f"  NRMSE (mean):  {metrics['nrmse_mean']:.2f}%")
print(f"  Geometric Mean Score: {metrics['geometric_mean_score']:.4f}")
print(f"  F1 Score: {metrics['f1_score']:.4f}")
print(f"\nBasic Metrics:")
print(f"  MAE:  {mae:.4f} kW")
print(f"  RMSE: {rmse:.4f} kW")
print(f"  R²:   {r2:.4f}")
print("=" * 80)

## 8. Visualizations

In [ ]:
# Scatter plot: Predictions vs Actual
plt.figure(figsize=(10, 6))
plt.scatter(targets, predictions_corrected, alpha=0.3, s=10)
plt.plot([targets.min(), targets.max()], [targets.min(), targets.max()], 'r--', lw=2, label='Perfect Prediction')
plt.xlabel('Actual Capacity (kW)', fontsize=12)
plt.ylabel('Predicted Capacity (kW)', fontsize=12)
plt.title('Site B: Predictions vs Actual', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
zoomed = False

if zoomed:
    sample_start = 525
    sample_end = 625
else:
    sample_start = 525
    sample_end = 900

plt.figure(figsize=(14, 6))
x = np.arange(sample_start, min(sample_end, len(targets)))
plt.plot(x, targets[sample_start:sample_end], label='Actual', alpha=0.7, linewidth=1.5)
plt.plot(x, predictions_corrected[sample_start:sample_end], label='Predicted', alpha=0.7, linewidth=1.5)
plt.xlabel('Sample Index', fontsize=12)
plt.ylabel('Capacity (kW)', fontsize=12)
plt.title(f'Site B: Time Series Predictions (Samples {sample_start}-{sample_end})', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
print(targets[600])
print(predictions[600])

In [ ]:
# After masking
df_val_masked = df_train_site_b[mask_site_b].reset_index(drop=True)

# Align with sequences
df_val_aligned = df_val_masked.iloc[sequence_length:sequence_length + len(predictions)].reset_index(drop=True)

# Check DR flags at sample 675
sample_index = 675
dr_flags = df_val_aligned.loc[sample_index, ['Demand_Response_Flag_-1', 'Demand_Response_Flag_0', 'Demand_Response_Flag_1']]
print(dr_flags)

if dr_flags['Demand_Response_Flag_1'] == 1:
    print("✅ There is a DR event at sample 675")
else:
    print("❌ No DR event at sample 675")


In [ ]:
# Error distribution
errors = targets - predictions

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(errors, bins=50, alpha=0.7, color='skyblue', edgecolor='black')
axes[0].axvline(x=0, color='red', linestyle='--', linewidth=2, label='Zero Error')
axes[0].set_xlabel('Prediction Error (kW)', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('Site B: Error Distribution', fontsize=14)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Box plot
axes[1].boxplot(errors, vert=True)
axes[1].set_ylabel('Prediction Error (kW)', fontsize=12)
axes[1].set_title('Site B: Error Box Plot', fontsize=14)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Error Statistics:")
print(f"  Mean Error: {errors.mean():.4f} kW")
print(f"  Std Error: {errors.std():.4f} kW")
print(f"  Min Error: {errors.min():.4f} kW")
print(f"  Max Error: {errors.max():.4f} kW")

## 9. Additional Analysis

In [ ]:
# Analyze by demand response flag
print("\nPerformance by Demand Response Flag:")
print("=" * 60)
for flag in [-1, 0, 1]:
    mask = val_demand_flags == flag
    if mask.sum() > 0:
        flag_mae = np.abs(targets[mask] - predictions[mask]).mean()
        flag_rmse = np.sqrt(np.mean((targets[mask] - predictions[mask]) ** 2))
        print(f"Flag {flag:2d}: {mask.sum():5d} samples | MAE = {flag_mae:7.4f} kW | RMSE = {flag_rmse:7.4f} kW")

In [ ]:
# Find worst predictions
abs_errors = np.abs(targets - predictions)
worst_indices = np.argsort(abs_errors)[-10:][::-1]

print("\nTop 10 Worst Predictions (Site B):")
print("=" * 80)
for idx in worst_indices:
    print(f"Index {idx:5d}: Actual={targets[idx]:7.2f} kW | Predicted={predictions[idx]:7.2f} kW | Error={abs_errors[idx]:7.2f} kW")

## 10. Summary

In [ ]:
print("\n" + "=" * 80)
print("EVALUATION SUMMARY - SITE B")
print("=" * 80)

print(f"\nModel: {MODEL_CHECKPOINT}")
print(f"Architecture: {MODEL_CONFIG['num_layers']} layers, {MODEL_CONFIG['hidden_size']} hidden units")
print(f"\nValidation Site: Site B")
print(f"Validation Samples: {len(predictions)}")
print(f"\nKey Metrics:")
print(f"  • NMAE (mean): {metrics['nmae_mean']:.2f}%")
print(f"  • NRMSE (mean): {metrics['nrmse_mean']:.2f}%")
print(f"  • MAE: {mae:.4f} kW")
print(f"  • RMSE: {rmse:.4f} kW")
print(f"  • F1 Score: {metrics['f1_score']:.4f}")
print(f"  • R² Score: {r2:.4f}")
print(f"  • Prediction Std: {predictions.std():.4f} kW")

if predictions.std() < 1.0:
    print(f"\n⚠️  WARNING: Model produces nearly horizontal predictions!")
elif metrics['nmae_mean'] < 30:
    print(f"\n✅ Model performance looks good on Site B!")
elif metrics['nmae_mean'] < 40:
    print(f"\n🔶 Model performance is acceptable on Site B, but could be improved.")
else:
    print(f"\n❌ Model performance needs improvement on Site B.")
    
print("=" * 80)